# Convolution as a matrix product: im2col forward and backward

Real libraries compute a convolution as one big matrix product. **im2col** ("image to columns") copies every window of the input into one row of a matrix; the convolution is then that matrix times the flattened kernels. The backward pass reverses the copy (**col2im**), adding up the gradients of positions that several windows share. Implement both, in NumPy without loops.

```python
def conv_forward(x, w, b, stride=1, padding=0):
    ...
    return out, cache

def conv_backward(dout, cache):
    ...
    return dx, dw, db
```

- `x`: `(N, C, H, W)`; `w`: `(F, C, k, k)` (F filters, square kernel); `b`: `(F,)`; `stride`, `padding`: ints (same for height and width; no dilation, no groups).
- `out`: `(N, F, H_out, W_out)`, equal to PyTorch's `F.conv2d(x, w, b, stride, padding)` (cross-correlation, zero padding). `cache` is anything you choose.
- `dout` has the shape of `out` and holds $\partial L/\partial\,\text{out}$. Return $\partial L/\partial x$, $\partial L/\partial w$, $\partial L/\partial b$ with the shapes of `x`, `w`, `b`. Padding positions are not part of `x`: their gradient is dropped.
- Do not modify any input.

The forward pass in three steps: pad; form the matrix `cols` of shape `(N * H_out * W_out, C * k * k)` whose rows are the flattened windows; compute `cols @ w.reshape(F, -1).T + b` and reshape to `(N, F, H_out, W_out)`. For the backward pass, write `dout` as a matrix `d` with the same row order as `cols`. Then $\partial L/\partial b$ sums `d` over rows, $\partial L/\partial w$ is `d.T @ cols` reshaped, and `d @ w.reshape(F, -1)` gives the gradient of every entry of `cols`, which must be added back to the input position it was copied from.

**Example.**

```python
x = np.arange(16.).reshape(1, 1, 4, 4)
w = np.ones((1, 1, 3, 3)); b = np.array([1.])
out, cache = conv_forward(x, w, b)            # [[[[46., 55.], [82., 91.]]]]
dx, dw, db = conv_backward(np.ones((1, 1, 2, 2)), cache)
dx[0, 0]   # [[1, 2, 2, 1], [2, 4, 4, 2], [2, 4, 4, 2], [1, 2, 2, 1]]: how many windows contain each pixel
db         # [4.]
```

**Restrictions.** Only `numpy` (any submodule). No Python loops or comprehensions. Banned: `convolve`, `correlate`, `fftconvolve`. Hint: `np.add.at(a, index_arrays, values)` adds `values` into `a` and, unlike `a[index] += values`, accumulates repeated indices.

**Constraints.** $N \le 8$, $C, F \le 32$, $H, W \le 32$, $1 \le k \le 5$, stride $\le 3$, padding $\le 2$; output at least 1 by 1. Time limit 2 s per call for `x` of shape `(8, 16, 32, 32)` with 32 filters of size 3, CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem6_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem6_submission.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def conv_forward(x, w, b, stride=1, padding=0):
    # Your code
    return out, cache


def conv_backward(dout, cache):
    # Your code
    return dx, dw, db


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem6_submission as submission
importlib.reload(submission)
# e.g. submission.conv_forward(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W41z3Lax/1fQy0xFOmdaOtlOI0d5T3Hk1G/8NbY7bUenYXBH3IkVj2QI0neKqv/9/XYBkOAdJVtO33NTiQSB/d7F7gK6HqXJ6EiMmoPJw3L/6WgsRitVSwxd736p0zpTNPa8yD8VWVOnRS6kFlKsZF2lG1FWRdLM6yORribzIhOLolrLKhEyT8RMzi/phQBlSusiB6S8yTJCmeZNrTQGDvfxmqSLRTpvsvoKI08woPK6oufRHIhjC5VpBb5MxblcMV3AP8vU6mmsm9kq1Rr00SyZZcVaJXG6KouqJjRno7xZlVejc3ydyTxXCQ/Oma9PihbNi6pSmaz5ZbGo2292TZwVRUmw6qpRJByl67hOV6po6rhlpZZLg2+e59G8ExtjwJAvFSYp0lmapPkyXqd5Uqy7caDUjFsXTTVndosqXaa5zGhSKat6Yqm5IYrTjAV6PbqQVQ55RyVLcMr/+6vKSlVpUV/IWlipiWUlExo0YhLBr4uqWAm73I0++DWMpvk0P7HTRaoF+KqhokdNntb60Q/068dHFireyzT58ZGZDSIi8bIWiVqkgCpIaGLR5HOSiRakx4QH4weGtlpeKjypad6pVBiti18x9GskTv9x8vrdq9MPIkt1TSwpA5aAaaEvinUusOh9k4vgU6pTUBU+Ex8IXC2qBmjVJ1Vd8SKw9kKmWVNhZSVTrQS9inVaX5CVQw5yqcS6SmswTOYtflL5kdDySqyJ3DW8YV0V+XIscoIqEuhOXKRJgulpXja1JukZLeRWpPOivGpfyIJIvvaVdU8+lpc0+o14XmRyJp6/+xvcrlJCk2VXJKucSNnT4vVEvKuKsdCFyIq5zAgi7EmLVQOhzBQJFSIqyMVSfM5SksMM/F9AlQs5r4sqIkyTaJ+UK8WyIf0HBrMGrEw9rC8qJROhSwWFYVZVNMuL7ArWki3g7kQFHIgEXSkSei3SBRGIV/ZBy4f1FxLKh3enpz/HL06ef3z7XhwTdmJ4ms8zCeykhuB0M1clWUp4NM0F/pX4ZGbBoCBxlQSwxWQsVnrp5gBvXpCQ88SO0L9OuQHN7YDEeRls3FpEHW+N01BRzS+80QX4T3Ndy3yugs3YfI8+qlwXVegtZ7SqbqpcbKIEQXZ+EYTRvGzwk9UchGayYjbFS0Z3WlVF5UExLHuw8jKSWlaVvALdHR/6QpYqWBb1GEaZ00/Y5/F0BMtGYJ2OHGUasq6bMlMBcY75YcRLLS0sU8xxkwhWOBaL6eiaAN5A4drgEtcacUdtSjWvYRPX3vwboOsom2eFvouysajqIjs+UA+fjoW0j39x9MKz1iC5JVbq+gqMLrJCEmE0zjj7HzxmloY/YmltHm9hxzBgp4dDvLlPzF6LAApJNcJbWmM16MiyIBSIFGSF/re1/eajp0gqU46EOa1J80UL3Boy6TvLrBAhDCsu+mHFRT98w1NVBYmxIIi4ldzQLznTwVI8FOswDAk2GE1/V0Jl8It9cr4BV+nLiQOd+CQzxAcRZLJaUthVZLDiGr+OosPlTdjTvfqtkdkXWGXfqcxUIjpP2NLHIUsURMi6ruz36YjV0QEZMBfPQLo5/+dm4ZsG0x9bOYC4HWfypdpCIK0cbVMMYyBqmfkOBoLxNT79qeqRRrMw1FNGqmN2EVYHPelb9cEIPY10S3bop1GIwgIOo5izszjukbNE+CPNWax7btKeg9qnNNa5LLGT1/+/oTkrcnXPoAyUzio3ZJK8t49CzoDJe/v0BEk6hyApcQl9evrM9agjgEGiVHn8EZmeZ2CWwI8Q3zZ5uwDC3hZCQxHB5G+bnuChlkCOxezr5S6/SO5mivELxve1Ypckdoaj+6HAIpIGiQ5m4VCswfIuzgwub+PEzD6ybvueTRyMTbSDZee+rhwchuAlL0W8auAUyG6CRT4WDxBO4RkPHlB5sNSOEpM6PscOIBY5Y15Qgko8IMdBFrhUVG4hZayWzQpJObYQUWZyriKXdfKuT/6EuHjWeZYMOZ+VtIAwnpuZyM5o88iDbYLMZ1qSwpDBsA5pqUIqoyqUTcHvaRmYNYyjZ+AcT1rjskHkCoUNbDFRlOCni5QyS0qZO16+uU7Ft+Lg5plYF9UlZfWSjVeQ+pCPtvHSChnEdxKmJD7NA60oFQRVXyrlZ76IEV39lHsgk+ZiE/uhRu4NDkhAnOvutSuRvJsailC9BYiKSjAUSCtNLHEe/lrOxRwIaAmYNFk3VSuc2FNygOri4axJEElpmHNwpOUpFVBNCXcS9TqdKwIICaXYoYuF9WDQ7CSq2YRoZIF8ibZvgoAVDb5F4iN9SCsMg5lsj1DQvmCQkYRV0jOrep+ySUCIUF0uYgajKhdtPmNKST28GDlKbdMRy++xsEoUj4RfNnQmGZMpVuQNwcS3OygRaH44tpC2QtEMNc1lNyQnY3E5AbZ+fGSSx1uDPU4+LwqmkwQxYTFMvGEWAyq2IKEt9hZx+PmmzxD7UV0UbDco/pP6KJosbpDlw8rUM1t1tUbMxgS3Y0Ciyamev7aiseu4hkRh51Q88I9RorJEGbyAaS0pgTVF5rXVEwMi77jTQat8Ce9UiVNYV+BAk0mxgrQXEuYXtxOn3EdptJRpEXOHodfl+IVG4AGgrGsepBw0LtIsIa+bU2cEBXNKvkAz/+fD2zegNWfP0XUCCtlXy6v6AiMPXwqHR/xgWxxxklY/ih86HPiId3jlv3ms/nGn5IezeeU/FLups3TWDplfGImaOs264aJ9/JcuuiZBKesLf7lOl7nslukrvdVecC8V9gXqQLUjsOKcmgWm16DnKdjMVRvuMHCZ1g8zhWniIHoaHdicTh9Rn2hFfb0xhxRUF3bzccGvlCmkvJzmDke0SDNYtHsNpiOQXVSKKkDbaUEmGj34W36ZUw9HU/etEgV3AIAwLUlr0QObKf719P0pPMfKInqH30EcUxssjsMIgYuWI6srZQXKpjmkEtHkCNaqqjrYhzMgfSAwIQAO9b64FyS+wV79mzwSp4/3J9wJe/X3k39+iH86efPm9GdQcD0dxbbVGMfEDBhz+jSvaqPm9gl5vnmaF6sStE5HN9P81du37+I3b38+/QBwAWwlelHAV+nh7zBcZR5P9FU+bz+8Qg75HDDM2wflvfyMJLN7+0XltD8X1emmDP0Wy0fTi/lclyWWqPRWAdlYs0K8wbalWo/lUtEB8rsRlNnM4/mFml/qQFdzUnEt+/vte+PvlA1jsyI5FpdGOCu9hGCeISxSBmh6fOw2ut5rtcPJkG0l0kZVV6lpK/a2KLMJnZ0PZbN1pZDYsZRgJaivQWg/D/1whfp8w3koNeXUbn54Zsk+grWgaGupP6IwqXm5LZGL3MSZaxXR77y4OaJnzEbxc2MJtK1r2hIgrwiBGX6y1c8mGZ2dW0Jnsh3nfbJ2Ww63Q+NGMzBvfAZ/iE2v1I6TiSNlhO+mCVyFMrHKJAoGMCCY+QFVNJR0pzNq34OdS87dVBaRYVsE3Cf3ZL6S9arJ/JEFd0cQqik2Q/hrmV0GpIytrdtL02m6sWdTGmxXFW0qSxMjJvdodwOzLRXky4HkSZSUr6i1YF4jYIMgKXMMVsg8EY6omCPgKybWKCI8Gt4bPV1EMkksjnB3cqcbb16kyywldRPOs4cH5wMLO+VFTZlQ1j2w1i8Ss7vF+AJRb5sbKBS6YjnaxjvY7xxqQJa8wgoSz39AitsSBLQtKXii60lgyAK2lt4mvK9aynzeV+gnzne2ed8yCKaDHO2LoL7xIvJdANPkdq6GJw1h7DarbZzs+JEsS5Ungev4mEXTkQl5Jrz/V19eQ1h4F/wpzd/aXeykWZ5o2oNC01vZWhAVdt5rWb9Gyrhj0yYG3ZM4EOYZpAeSkkQL6tbw77IA4yXG5GmUHYLmRf8qqDzlABt4eMJwOLp8SzsKBONAXe+NxZ4B4jyKHHAP+FD3LveoFXsz3E8cov8jH2vukK8d7R2wb9oTRDqjJNnOL+wOwalgt0Fw+1lJFLH/4N/hM1ONZmt5pR+ag1hUMsgL0xYQGaWFVuTZ1djhROXSbTWBsQLTRLKpFCUdVyJTS5Tma0qbBM+Moij0NjMkZSIpSCkG8EXK9aXRQuBtmn/mzdHbhQ25dvOFGf5bBN5e+mfRyws9EwKGe5sOKNDCyofgBzocsB1ADv+IhvPC4WiPZDstg/I+87FxorgoTT7Z77bRdONkW673JQxTRQm7+G/hYFPJ6vmNCEzmZDGc7Z/7xj3A/n1E0GG9i3kObjss8+jXMWzSJDb2HqMYJjcmo67Uhcq5iG1zR171n+XfcnZzR6W+qFKAya5izmQDgEChG1M11c/o3+bqIdOZKD2vUq4qRAlThX8vhT2J5gmpO76mgg3uqmie5sP6fodpRm0VV7dGqJvxXMf1LFBRHLcfUPLZZJOAI9Nc8Ka+IDwAAU0tqPxULuejyq9lwWala2qZUMOKoiz3Jq3R4SelY17W3hsiVbDB0Gw+TuvIt8KkExM+IVHe+Qhl/zfXjPWG5jthr5CJB20thki7XeVSJSur5aezNkV0rNw6c3LulcRWUpSAH4t2yqFL12H3GKeGQ5QVMtFBwEQ8IrMhn+BWxIjgYc+iTkbggh3KJ4JoifEnOGlwN/FY7PqFKRhp6Oyc3qnU092racwYk7WVNFkiDbwpoAIf/lkH7ZzIuaUUbR2dMlhyQX1mqDrn3YLe2EvO/XTWlXQun9Xcd9xB60cJ7loELE66HKIDM93f5Y2VuJbnAkSnRfShpvbJy7dOeKbJE5lfgX378PKXk1fvX49tlW6nIpzF3Nx1p8BeLOtfWyJZTvYdLf3qmC/BdM0qqDNJKwXnM8oIQOl2muVoVHVKKCpH5suPL1+fvo/fn568GrfUbaU5ulRkPf1WWESjMfVmTGeHupN8ZOJx5N0JM6bRNeagJi9UbeFrZrvoLCxGSKgD+rG9EB5j5ukz+swFw7kx/F2W2IdUFVEPKDbLKPKEXyG4/X5vwrZdei0Ja4jWOc5NNDM8UgTmqMYRty6KSz5TENdOH0fRPnWAXeiyWH6SWrUdItMD6VpiP7063d8/6Ai4Fxu3UWwPhphMOomhG2KFVZRNgu7Yj1q33gE9JCgXVwgv7Jksik/iAxTgj/f394/Ov9qR+Q5jv4/DQ/79DjPHnRW7E03wMjafws8rt5NVUiiTNJnLdthdCMSNX7ffk4Xl/XxyOjKX/sgL2y2jvQfY8m3vEX7e95ae8y1v8yUDzc6y1/1sY8tVeWYKRQZ3ddAl8Tbs0V5DSUOAQmVhAjs9UWz/JCvtcERprVY64NszeX9X4Kjqzv3pAEWCjGAR+g0vRAk6hOSUhDBuJc+8HZP1Vw3iljEKylb4FkHuWNsKuBSe07xRvv7mZoM18fCIgZw9OTpH/OZzYVBru9SCw+N0ZEG7yR42+krdYCp7hsoUG2/NoQ8N7Uf7N/c5E9u9+PAVu84f3Xns4dxuVP6qkGaVcNZJjjyVEu/duxs7EbxbbHIP4+McrLMiX7pzme2YLYI0N7fMOJsfUwGxkJVwJ4OhHwQscj7c2Opp/0GGW5opvR64q3KvveQ/RYt/z4DPK5Iv3EM6WK2BE7wKVpwEtx3UjsXh0PZm09lzV5BVbWC791bULjC56jGVjkGXu96VlZp7+UWRBTuEmW8Equ5ZLoGre+DaFa5AuXVL4WvNC+EKHhPd4pjqGzooO3LHA1TumJPddrOgM92hG9H9+0f+W5TnketiSDbsF7cf6o35AHxsrqaO/bs4Y3ttZEyn08RAe9sc+4ON82ojVyXFy3PvoqO96ka3vNZ8GUgj3owF6dTFqw3df8T/Z5SYB4bqmu9I0aWYhArEYzPKWfvTxyBC/dYg/Gk+7Da3irputcMV9u5bvOA/O5iAkhYdn3SmiTommmRCf25wXHaL2j9K6NNEpA+SFe50Cro7dPZi8xjMRkRzN7DeHpj1B7x7AU1uciDsogq66kR6fAAGjunkFuo73u8uDmg+sehuVGF55P/tiA8EIMLhW44V3U3kS53GHTKV01hIljsx9x6ZJJTuPnBz095KI2CpzSENFbYpTzdGAUTZS2wXBdnBxtwqQ4mOoDQRD0RJl3TbwVA8egSH/lbY0LjuLTocWnS4sygxptHeo4jyolpxFfm7Om6B7aPUXnvPFwXeC6dtulEaMyP8lGzcw9o9UD11px8MS51henfwWPRkVN0FXYvav6frFEGcebd1AcfchCd8OzdiOyDdZV1zs7pjjJ+2MPiX1L/rLql/72W02xaYydUskSI5Eq0ttm6WOAO5Uy4MdcceeZQt8nDAIh2KvkmSrkhNyawzyS4lXRrGKaLQFT5EuWRjErtkbX/PpvSXSJags1v1f75z5c9nZ1fJy76Kh9R7TTTeDGp4uavfbd12+l2OvYvTfeB3KNdFJD/sc57oAo+LVHTHw24KY3M3lO+iHTyNuO/G9CJ24T8E9cchzymwKwV29JDyhnDcXisNzg6i83CbArrvmatMxxBebG7TaJ+cpfXxg3CHPPsnSXNkYEfiUDhIIEZYQET5sh8YJkzY07H4jmjb+vjYUG0J738kDjlEf46Sg82BuFTYn7Nh9E+swHbRH/JHkt7ux0OHfrItQbMPsvzsRjgkv8lnqDZQaEuwQBBnB8nHf98NS++QPw5Lj8ifMG/MxOGXUXOI8CA/YScs0w00K5q80ea4agv1AUP/y1h8P0CXIZr+2/04CZlit/0+/gxdTzZPrG47OU1utbLvYWi3Whk0/WTYygwzTM+THWWX/MdMu/p1ErV7FEa3sEI4BzD7QyJtMqS9iZ0wrL/J0LZvryNPov2xuCM9YYvubyuDK12kHw/Qbuh2tNtcpPcHSzaCEgJs9bydbLzwy3/bY9WYbIajr1vbC8IbPwKPbm7+Fwu+e3A=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
